# Self-Consistency

Generate three separately queued paths and majority-vote normalized answers.

This executed notebook uses only public prompts, responses, and aggregate analysis. It does not embed answer keys or row-level correctness labels.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
manifest_path = root / 'tier1_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

def read_public_csv(path):
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, keep_default_na=False)
    forbidden = {'gold', 'ground_truth', 'correct_answer', 'normalized_gold', 'is_correct'}
    exposed = forbidden.intersection(str(column).lower() for column in frame.columns)
    if exposed:
        raise ValueError(f'public analysis contains forbidden row-level columns: {sorted(exposed)}')
    return frame

print('model:', manifest.get('model', 'not prepared'))
print('reasoning effort:', manifest.get('reasoning_effort', 'not prepared'))
print('prompt version:', manifest.get('prompt_version', 'not prepared'))


model: gpt-6-sol
reasoning effort: medium
prompt version: gpt6-tier1-v1


In [2]:
condition = 'self_consistency'
queues = ('self_consistency_path_1', 'self_consistency_path_2', 'self_consistency_path_3')
coverage = []
result_frames = []
for queue in queues:
    dataset_path = root / 'Dataset' / f'{queue}.csv'
    result_path = root / 'Result' / f'{queue}.csv'
    dataset = read_public_csv(dataset_path)
    result = read_public_csv(result_path)
    answered = 0
    if not result.empty and 'model_response' in result:
        answered = int(result['model_response'].astype(str).str.strip().ne('').sum())
        result_frames.append(result.assign(source_queue=queue))
    coverage.append({
        'queue': queue,
        'dataset_rows': len(dataset),
        'result_rows': len(result),
        'answered_rows': answered,
        'unresolved_rows': max(len(result) - answered, 0),
    })

display(pd.DataFrame(coverage))
if result_frames:
    results = pd.concat(result_frames, ignore_index=True)
    group_columns = [column for column in ('task', 'variant', 'stage', 'path_index') if column in results]
    display(
        results.groupby(group_columns, dropna=False)
        .agg(responses=('response_id', 'count'), answered=('model_response', lambda values: values.astype(str).str.strip().ne('').sum()))
        .reset_index()
    )
else:
    print('Result files are not available yet.')

analysis_dir = root / 'Analysis'
shown = 0
if analysis_dir.exists():
    for path in sorted(analysis_dir.glob('*.csv')):
        frame = read_public_csv(path)
        if frame.empty:
            continue
        selector = None
        if 'condition' in frame.columns:
            selector = frame['condition'].astype(str).eq(condition)
        elif 'method' in frame.columns:
            selector = frame['method'].astype(str).eq(condition)
        if selector is not None and bool(selector.any()):
            print(path.name)
            selected = frame.loc[selector].reset_index(drop=True)
            if 'evaluation_id' in selected.columns and len(selected) > 100:
                print('public row-level aggregate rows:', len(selected))
                grouping = [column for column in ('task', 'variant', 'agreement', 'status', 'answer_changed') if column in selected]
                if grouping:
                    display(selected.groupby(grouping, dropna=False).size().rename('rows').reset_index())
            else:
                display(selected)
            shown += 1
if not shown:
    print('No condition-level public analysis tables are available yet.')


,queue,dataset_rows,result_rows,answered_rows,unresolved_rows
0,self_consistency_path_1,450,450,450,0
1,self_consistency_path_2,450,450,450,0
2,self_consistency_path_3,450,450,450,0


,task,variant,stage,path_index,responses,answered
0,obfus_blood_relation,base,path_1,1,50,50
1,obfus_blood_relation,base,path_2,2,50,50
2,obfus_blood_relation,base,path_3,3,50,50
3,obfus_blood_relation,obfuscation_l1,path_1,1,50,50
4,obfus_blood_relation,obfuscation_l1,path_2,2,50,50
5,obfus_blood_relation,obfuscation_l1,path_3,3,50,50
6,obfus_blood_relation,obfuscation_l2,path_1,1,50,50
7,obfus_blood_relation,obfuscation_l2,path_2,2,50,50
8,obfus_blood_relation,obfuscation_l2,path_3,3,50,50
9,obfus_direction_sense,base,path_1,1,50,50


accuracy_by_cell.csv


,model,condition,task,task_title,variant,correct,total,accuracy_pct,response_or_consensus_failures
0,gpt-6-sol,self_consistency,obfus_fol,FOL,base,40,50,80.0,0
1,gpt-6-sol,self_consistency,obfus_fol,FOL,obfuscation,32,50,64.0,0
2,gpt-6-sol,self_consistency,obfus_blood_relation,Blood Relation,base,39,50,78.0,1
3,gpt-6-sol,self_consistency,obfus_blood_relation,Blood Relation,obfuscation_l1,33,50,66.0,2
4,gpt-6-sol,self_consistency,obfus_blood_relation,Blood Relation,obfuscation_l2,30,50,60.0,2
5,gpt-6-sol,self_consistency,obfus_number_series,Number Series,base,42,50,84.0,1
6,gpt-6-sol,self_consistency,obfus_number_series,Number Series,obfuscation,41,50,82.0,2
7,gpt-6-sol,self_consistency,obfus_direction_sense,Direction Sense,base,27,50,54.0,8
8,gpt-6-sol,self_consistency,obfus_direction_sense,Direction Sense,obfuscation,24,50,48.0,10


base_obfuscated_gaps.csv


...

,condition,task,variant,method_correct,control_correct,total,method_accuracy_pct,control_accuracy_pct,improvement_pp
0,self_consistency,obfus_fol,obfuscation,32,32,50,64.0,64.0,0.0
1,self_consistency,obfus_blood_relation,obfuscation_l1,33,34,50,66.0,68.0,-2.0
2,self_consistency,obfus_blood_relation,obfuscation_l2,30,36,50,60.0,72.0,-12.0
3,self_consistency,obfus_number_series,obfuscation,41,38,50,82.0,76.0,6.0
4,self_consistency,obfus_direction_sense,obfuscation,24,20,50,48.0,40.0,8.0


paired_retention.csv


,condition,task,variant,both_correct,base_only_correct,obfuscated_only_correct,both_wrong,total,paired_retention_pct
0,self_consistency,obfus_fol,obfuscation,30,10,2,8,50,75.0000
1,self_consistency,obfus_blood_relation,obfuscation_l1,33,6,0,11,50,84.6154
2,self_consistency,obfus_blood_relation,obfuscation_l2,30,9,0,11,50,76.9231
3,self_consistency,obfus_number_series,obfuscation,40,2,1,7,50,95.2381
4,self_consistency,obfus_direction_sense,obfuscation,22,5,2,21,50,81.4815


self_consistency_aggregates.csv
public row-level aggregate rows: 450


,task,variant,agreement,status,rows
0,obfus_blood_relation,base,majority,answered,4
1,obfus_blood_relation,base,none,no_consensus,1
2,obfus_blood_relation,base,unanimous,answered,45
3,obfus_blood_relation,obfuscation_l1,majority,answered,12
4,obfus_blood_relation,obfuscation_l1,none,no_consensus,2
5,obfus_blood_relation,obfuscation_l1,unanimous,answered,36
6,obfus_blood_relation,obfuscation_l2,majority,answered,15
7,obfus_blood_relation,obfuscation_l2,none,no_consensus,2
8,obfus_blood_relation,obfuscation_l2,unanimous,answered,33
9,obfus_direction_sense,base,majority,answered,7
